# Qwen 27B — Kaggle-এ ফোনের ব্রাউজার থেকে চালানোর চ্যাট ওয়েবসাইট

**আগে:** Settings → Accelerator → **GPU T4 ×2**, আর **Internet ON** করুন।

সেলগুলো ওপর থেকে নিচে একটার পর একটা চালান। একদম শেষ সেল একটা লিঙ্ক দেবে — সেটা ফোনের Chrome-এ খুলুন।

- মডেল দুই T4-তে ভাগ হয়ে পুরোটা GPU মেমরিতে থাকে (`OLLAMA_SCHED_SPREAD`), CPU-তে কিছু নামে না
- Flash Attention + q8 KV cache: কম মেমরিতে দ্রুত
- ফোন থেকে ছবি/PDF/DOCX/XLSX/টেক্সট/কোড ফাইল পাঠিয়ে বিশ্লেষণ
- লিঙ্কে গোপন কী (`?k=...`) আছে, তাই লিঙ্ক না জানলে কেউ ঢুকতে পারবে না
- **নতুন:** মডেল আগে থেকেই GPU-তে লোড হয়ে পিন থাকে (কনটেক্সট অটো-বাছাই), প্রতি ২ মিনিটে গরম রাখা হয়
- **নতুন UI:** রঙিন আধুনিক ডিজাইন, বাঁ পাশের মেনুতে আগের চ্যাটের তালিকা, লেখার সময় জোর করে নিচে টানে না


## ১. GPU দেখুন + দরকারি প্যাকেজ

In [1]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip -q install fastapi uvicorn httpx python-multipart pypdf python-docx openpyxl
!apt-get update -qq && apt-get install -y -qq zstd
!curl -fsSL https://ollama.com/install.sh | sh
!ollama --version

name, memory.total [MiB]
Tesla T4, 15360 MiB
Tesla T4, 15360 MiB
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package zstd.
(Reading database ... 130198 files and directories currently installed.)
Preparing to unpack .../zstd_1.5.5+dfsg2-2build1.1_amd64.deb ...
Unpacking zstd (1.5.5+dfsg2-2build1.1) ...
Setting up zstd (1.5.5+dfsg2-2build1.1) ...
Processing triggers for man-db (2.12.0-4build2) ...
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%###############################################       93.9%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434

## ২. সেটিংস (দরকার হলে এখানে বদলান)

In [ ]:
import os, subprocess, shutil, time, json, urllib.request, secrets
from pathlib import Path

SOURCE_MODEL = "hf.co/JonathanColetti/Qwen3.8-27B-Uncensored-GGUF:Q4_K_M"
NUM_CTX = 32768          # 32K: দ্রুত + নিরাপদে ৩০GB-তে ধরে। পুরো GPU-তে ঢুকলে 49152/65536 চেষ্টা করতে পারেন
VISION_HELPER = "qwen2.5vl:3b"   # মডেলের নিজের ছবি-দেখার ক্ষমতা না থাকলে শুধু তখনই নামবে
NUM_CTX_FALLBACKS = [32768, 24576, 16384, 12288]   # পুরোটা GPU-তে না ঢুকলে সেল ৬ নিজেই ছোটটা চেষ্টা করবে
PORT = 7860

# /kaggle/temp-এ বেশি জায়গা থাকে; না থাকলে /kaggle/working
base = Path("/kaggle/temp") if Path("/kaggle/temp").exists() else Path("/kaggle/working")
MODELS_DIR = base / "ollama-models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

OLLAMA_URL = "http://127.0.0.1:11434"
OLLAMA_BIN = shutil.which("ollama")
OLLAMA_ENV = os.environ.copy()
OLLAMA_ENV.update({
    "OLLAMA_HOST": "127.0.0.1:11434",
    "OLLAMA_MODELS": str(MODELS_DIR),
    "OLLAMA_KEEP_ALIVE": "-1",
    "OLLAMA_FLASH_ATTENTION": "1",
    "OLLAMA_KV_CACHE_TYPE": "q8_0",
    "OLLAMA_NUM_PARALLEL": "1",
    "OLLAMA_MAX_LOADED_MODELS": "2",
    "OLLAMA_SCHED_SPREAD": "1",      # দুই GPU-তে সমানভাবে ভাগ
    "OLLAMA_LOAD_TIMEOUT": "15m",
    "OLLAMA_NOPRUNE": "1",
})
print("models dir:", MODELS_DIR)
!df -h {base} | tail -1

## ৩. Ollama সার্ভার চালু

In [4]:
def ollama_ready():
    try:
        urllib.request.urlopen(f"{OLLAMA_URL}/api/tags", timeout=2).close(); return True
    except Exception:
        return False

if not ollama_ready():
    log = open("/tmp/ollama-server.log", "a")
    ollama_proc = subprocess.Popen([OLLAMA_BIN, "serve"], env=OLLAMA_ENV, stdout=log, stderr=subprocess.STDOUT)
    for _ in range(60):
        if ollama_ready(): break
        time.sleep(1)
    else:
        raise RuntimeError("Ollama চালু হয়নি। /tmp/ollama-server.log দেখুন")
print("✅ Ollama ready")

✅ Ollama ready


## ৪. মডেল ডাউনলোড (প্রায় ১৭ GB)

In [5]:
r = subprocess.run([OLLAMA_BIN, "pull", SOURCE_MODEL], env=OLLAMA_ENV)
if r.returncode != 0:
    print("সাধারণ pull ব্যর্থ, --insecure দিয়ে আবার চেষ্টা...")
    subprocess.run([OLLAMA_BIN, "pull", "--insecure", SOURCE_MODEL], env=OLLAMA_ENV, check=True)
subprocess.run([OLLAMA_BIN, "list"], env=OLLAMA_ENV)

pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest 
pulling 4c5e2db039e9:   0% ▕                  ▏  33 KB/ 16 GB                  pulling manifest 
pulling 4c5e2db039e9:   0% ▕                  ▏ 1.5 MB/ 16 GB                  pulling manifest 
pulling 4c5e2db039e9:   0% ▕                  ▏ 7.5 MB/ 16 GB                  pulling manifest 
pulling 4c5e2db039e9:   0% ▕                  ▏  64 MB/ 16 GB                  pulling manifest 
pulling 4c5e2db039e9:   1% ▕                  ▏ 131 MB/ 16 GB                  pulling manifest 
pulling 4c5e2db039e9:   1% ▕                  ▏ 159 MB/ 16 GB                  pulling manifest 
pulling 4c5e2db039e9:   1% ▕                  ▏ 227 MB/ 16 GB                  pulling manifest 
pulling 4c5e2db039e9:   2% ▕                  ▏ 260 MB/ 16 GB                  pulling manifest 
pulling 4c5e2db039e9:   2% ▕       

NAME                                                        ID              SIZE     MODIFIED               
hf.co/JonathanColetti/Qwen3.8-27B-Uncensored-GGUF:Q4_K_M    2859adf8326a    17 GB    Less than a second ago    


CompletedProcess(args=['/usr/local/bin/ollama', 'list'], returncode=0)

## ৫. মডেলের ছবি দেখার ক্ষমতা আছে কিনা পরীক্ষা

না থাকলে ছোট একটা সহায়ক ভিশন মডেল (~৩ GB) নামবে, যে ছবি বর্ণনা করে দেবে আর মূল মডেল সেই বর্ণনা পড়ে উত্তর দেবে।

In [7]:
def api(path, payload, timeout=600):
    req = urllib.request.Request(f"{OLLAMA_URL}{path}", data=json.dumps(payload).encode(),
                                 headers={"Content-Type": "application/json"}, method="POST")
    with urllib.request.urlopen(req, timeout=timeout) as resp:
        return json.loads(resp.read())

caps = api("/api/show", {"model": SOURCE_MODEL}).get("capabilities") or []
print("মডেলের capabilities:", caps)
NATIVE_VISION = "vision" in caps
if NATIVE_VISION:
    print("✅ মডেল নিজেই ছবি বুঝতে পারে")
    HELPER = ""
else:
    print("ℹ️ মডেলে ছবি-সাপোর্ট নেই, সহায়ক ভিশন মডেল নামাচ্ছি:", VISION_HELPER)
    subprocess.run([OLLAMA_BIN, "pull", VISION_HELPER], env=OLLAMA_ENV, check=True)
    HELPER = VISION_HELPER

মডেলের capabilities: ['tools', 'thinking', 'completion', 'vision']
✅ মডেল নিজেই ছবি বুঝতে পারে


## ৬. মডেলকে আগে থেকেই GPU-তে লোড করে পিন করুন (এটাই দ্রুত রিপ্লাইয়ের চাবিকাঠি)

এই সেল নিজে থেকে সবচেয়ে বড় কনটেক্সট বেছে নেয় যেটায় মডেল **১০০% GPU**-তে ঢোকে। CPU-তে কিছু নামলে রিপ্লাই খুব ধীর হয়, তাই এটা জরুরি।

In [ ]:
def gpu_ps():
    return subprocess.run([OLLAMA_BIN, "ps"], env=OLLAMA_ENV, capture_output=True, text=True).stdout

def unload_all():
    for mdl in (SOURCE_MODEL,):
        try: api("/api/generate", {"model": mdl, "keep_alive": 0}, timeout=120)
        except Exception: pass
    time.sleep(2)

candidates = [NUM_CTX] + [c for c in NUM_CTX_FALLBACKS if c < NUM_CTX]
chosen = None
for ctx in candidates:
    unload_all()
    t = time.time()
    api("/api/generate", {"model": SOURCE_MODEL, "prompt": "hi", "stream": False, "keep_alive": -1,
                          "think": False, "options": {"num_ctx": ctx, "num_predict": 4}})
    print(f"num_ctx={ctx}: লোড হতে লাগল {time.time()-t:.0f}s")
    ps = gpu_ps(); print(ps)
    if "100% GPU" in ps:
        chosen = ctx; break
    print(f"⚠️ {ctx}-এ পুরোটা GPU-তে ঢোকেনি, ছোট কনটেক্সট চেষ্টা করছি…")

if chosen is None:
    chosen = candidates[-1]
    print("⚠️ কোনোটাতেই ১০০% GPU হয়নি; সবচেয়ে ছোটটা ব্যবহার হচ্ছে:", chosen)
NUM_CTX = chosen
print(f"✅ ব্যবহৃত NUM_CTX = {NUM_CTX} (মডেল GPU-তে পিন করা, keep_alive=-1)")
!nvidia-smi --query-gpu=index,memory.used,memory.total --format=csv

# গতি মাপা (এই ওয়ার্ম-আপ রিকোয়েস্টে মডেল পুরো গরম হয়ে যায়)
res = api("/api/generate", {"model": SOURCE_MODEL, "stream": False, "keep_alive": -1, "think": False,
    "prompt": "Explain in about 150 words why GPUs are good for running LLMs.",
    "options": {"num_ctx": NUM_CTX, "temperature": 0}})
print(f"গতি: {res['eval_count']/(res['eval_duration']/1e9):.1f} token/s")


Apps Shell****

In [13]:
import os
os.makedirs("/kaggle/working/chatapp", exist_ok=True)

## ৭. ওয়েব অ্যাপের ফাইল তৈরি

In [ ]:
%%writefile /kaggle/working/chatapp/index.html
<!doctype html>
<html lang="bn" data-theme="dark"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1,viewport-fit=cover"><title>QWEN AI</title><style>body{background:#0b0f14;color:#e8eef5;font-family:system-ui,sans-serif;display:grid;place-items:center;min-height:100vh;margin:0;text-align:center}button{background:#7c5cff;color:#fff;border:0;border-radius:50px;padding:10px 22px;font-size:15px;margin-top:14px;cursor:pointer}</style></head><body><div id="l">QWEN AI loading…</div><script>
const CDN="https://cdn.jsdelivr.net/gh/sweetxhasan/qwen-ai-ui@main/index.html";
const B64="...";
async function show(h){document.open();document.write(h);document.close();}
async function fromCDN(){const r=await fetch(CDN,{cache:"no-store"});if(!r.ok)throw new Error("CDN "+r.status);const h=await r.text();if(h.length<100000||!h.includes("QWEN AI"))throw new Error("CDN bad");return h;}
async function fromEmbed(){const buf=Uint8Array.from(atob(B64),c=>c.charCodeAt(0));const s=new Blob([buf]).stream().pipeThrough(new DecompressionStream("gzip"));return await new Response(s).text();}
(async()=>{try{await show(await fromCDN());}catch(e){try{await show(await fromEmbed());}catch(e2){document.getElementById("l").innerHTML="Load hoyni, net check kore abar try koro.<br><small>"+e2.message+"</small><br><button id=\"rt\">Retry</button>";document.getElementById("rt").onclick=()=>location.reload();}}})();
</script></body></html>


In [ ]:
%%writefile /kaggle/working/chatapp/server.py
import hashlib
import io
import json
import os

import httpx
from fastapi import FastAPI, File, Request, UploadFile
from fastapi.responses import HTMLResponse, JSONResponse, PlainTextResponse, StreamingResponse

OLLAMA = "http://127.0.0.1:11434"
MODEL = os.environ["CHAT_MODEL"]
VISION_HELPER = os.environ.get("VISION_HELPER", "")
TOKEN = os.environ["APP_TOKEN"]
NUM_CTX = int(os.environ.get("NUM_CTX", "32768"))
SYSTEM = "You are a helpful assistant. Always reply in the same language the user writes in."

HERE = os.path.dirname(os.path.abspath(__file__))
HTML = open(os.path.join(HERE, "index.html"), encoding="utf-8").read()

app = FastAPI()
_vision = {"v": None}
_desc = {}


@app.middleware("http")
async def auth(request: Request, call_next):
    qk = request.query_params.get("k")
    k = qk or request.cookies.get("k")
    if k != TOKEN:
        return PlainTextResponse("Unauthorized", status_code=401)
    resp = await call_next(request)
    if qk == TOKEN:
        resp.set_cookie("k", TOKEN, max_age=3 * 86400, httponly=True, samesite="lax")
    return resp


async def has_vision():
    if _vision["v"] is None:
        try:
            async with httpx.AsyncClient(timeout=30) as c:
                r = (await c.post(f"{OLLAMA}/api/show", json={"model": MODEL})).json()
            _vision["v"] = "vision" in (r.get("capabilities") or [])
        except Exception:
            _vision["v"] = False
    return _vision["v"]


@app.get("/", response_class=HTMLResponse)
async def index():
    return HTML


@app.get("/api/info")
async def info():
    return {"model": MODEL, "vision": await has_vision() or bool(VISION_HELPER)}


@app.post("/api/extract")
async def extract(file: UploadFile = File(...)):
    name = file.filename or "file"
    data = await file.read()
    ext = name.lower().rsplit(".", 1)[-1] if "." in name else ""
    try:
        if ext == "pdf":
            from pypdf import PdfReader
            text = "\n".join((p.extract_text() or "") for p in PdfReader(io.BytesIO(data)).pages)
        elif ext == "docx":
            import docx
            d = docx.Document(io.BytesIO(data))
            text = "\n".join(p.text for p in d.paragraphs)
        elif ext in ("xlsx", "xlsm"):
            import openpyxl
            wb = openpyxl.load_workbook(io.BytesIO(data), read_only=True, data_only=True)
            lines = []
            for ws in wb.worksheets:
                lines.append(f"## {ws.title}")
                for row in ws.iter_rows(values_only=True):
                    lines.append("\t".join("" if v is None else str(v) for v in row))
            text = "\n".join(lines)
        else:
            text = data.decode("utf-8", errors="replace")
    except Exception as e:
        return JSONResponse({"error": str(e)}, status_code=400)
    limit = int(NUM_CTX * 2.2)
    return {"name": name, "text": text[:limit], "truncated": len(text) > limit}


@app.post("/api/chat")
async def chat(request: Request):
    body = await request.json()
    msgs = body.get("messages", [])
    think = bool(body.get("think", False))

    async def gen():
        vision = await has_vision()
        out = []
        async with httpx.AsyncClient(timeout=None) as c:
            # Images: send natively if the model has vision, otherwise describe them with a small helper model.
            for m in msgs:
                imgs = m.get("images") or []
                if imgs and not vision:
                    notes = []
                    for im in imgs:
                        h = hashlib.sha1(im.encode()).hexdigest()
                        if h not in _desc:
                            if not VISION_HELPER:
                                _desc[h] = "(no vision model available)"
                            else:
                                yield json.dumps({"status": "ছবি বিশ্লেষণ হচ্ছে…"}) + "\n"
                                r = await c.post(f"{OLLAMA}/api/chat", json={
                                    "model": VISION_HELPER, "stream": False, "keep_alive": "2m", "options": {"num_gpu": 0},
                                    "messages": [{"role": "user", "images": [im], "content":
                                        "Describe this image in detail. Transcribe all visible text exactly. "
                                        "Mention objects, people, layout, numbers and anything notable."}],
                                })
                                _desc[h] = r.json().get("message", {}).get("content", "")
                        notes.append(_desc[h])
                    extra = "\n\n".join(f"[Attached image {i + 1} - analysis by a vision model]\n{d}" for i, d in enumerate(notes))
                    m = {"role": m["role"], "content": m.get("content", "") + "\n\n" + extra}
                out.append(m)

            payload = {
                "model": MODEL,
                "messages": [{"role": "system", "content": SYSTEM}] + out,
                "stream": True,
                "keep_alive": -1,
                "think": think,
                "options": {"num_ctx": NUM_CTX, "temperature": 0.7},
            }
            for attempt in range(2):
                async with c.stream("POST", f"{OLLAMA}/api/chat", json=payload) as r:
                    if r.status_code != 200:
                        err = (await r.aread()).decode("utf-8", "replace")
                        if attempt == 0 and "think" in err.lower():
                            payload.pop("think", None)  # model does not support the think flag
                            continue
                        yield json.dumps({"error": err[:500]}) + "\n"
                        return
                    async for line in r.aiter_lines():
                        if line:
                            yield line + "\n"
                    return

    return StreamingResponse(gen(), media_type="application/x-ndjson",
                             headers={"Cache-Control": "no-cache", "X-Accel-Buffering": "no"})


## ৮. সার্ভার + পাবলিক লিঙ্ক চালু (এটাই শেষ ধাপ)

এই সেলটা চলতে থাকবে — **বন্ধ করবেন না**। নিচে লিঙ্ক ছাপা হলে ফোনের Chrome-এ খুলুন।

In [ ]:
import re, queue, threading

# cloudflared ইনস্টল
if not shutil.which("cloudflared"):
    !curl -fsSL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /usr/local/bin/cloudflared
    !chmod +x /usr/local/bin/cloudflared

APP_TOKEN = secrets.token_urlsafe(16)
app_env = os.environ.copy()
app_env.update({"CHAT_MODEL": SOURCE_MODEL, "VISION_HELPER": HELPER, "APP_TOKEN": APP_TOKEN, "NUM_CTX": str(NUM_CTX)})

if "app_proc" in globals() and app_proc.poll() is None:
    app_proc.terminate(); time.sleep(1)
app_log = open("/tmp/chatapp.log", "w")
app_proc = subprocess.Popen(["python", "-m", "uvicorn", "server:app", "--host", "127.0.0.1", "--port", str(PORT)],
                            cwd="/kaggle/working/chatapp", env=app_env, stdout=app_log, stderr=subprocess.STDOUT)
for _ in range(30):
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{PORT}/api/info?k={APP_TOKEN}", timeout=2).close(); break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("ওয়েব সার্ভার চালু হয়নি: " + open("/tmp/chatapp.log").read()[-1500:])

if "tunnel_proc" in globals() and tunnel_proc.poll() is None:
    tunnel_proc.terminate(); time.sleep(1)
tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--no-autoupdate", "--protocol", "http2", "--edge-ip-version", "4",
     "--url", f"http://127.0.0.1:{PORT}"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
q = queue.Queue()
threading.Thread(target=lambda: [q.put(l) for l in iter(tunnel_proc.stdout.readline, "")], daemon=True).start()
PUBLIC_URL, deadline = None, time.monotonic() + 60
while time.monotonic() < deadline and not PUBLIC_URL:
    try:
        m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", q.get(timeout=1))
        if m: PUBLIC_URL = m.group(0)
    except queue.Empty:
        pass
if not PUBLIC_URL:
    raise RuntimeError("লিঙ্ক পাওয়া যায়নি, সেলটা আবার চালান")

print("\n" + "=" * 60)
print("📱 ফোনের Chrome-এ এই লিঙ্কটা খুলুন:\n")
print(f"{PUBLIC_URL}/?k={APP_TOKEN}")
print("\n⚠️ লিঙ্কটা কাউকে দেবেন না — যার কাছে আছে সে আপনার AI ব্যবহার করতে পারবে।")
print("=" * 60)

# মডেলকে সবসময় GPU-তে গরম রাখা (আনলোড হলে ফেরত লোড করে)
def keep_warm():
    while True:
        try:
            api("/api/generate", {"model": SOURCE_MODEL, "keep_alive": -1, "options": {"num_ctx": NUM_CTX}}, timeout=900)
        except Exception:
            pass
        time.sleep(120)
threading.Thread(target=keep_warm, daemon=True).start()

# সেশন জাগিয়ে রাখা
while True:
    time.sleep(30)
    if app_proc.poll() is not None or tunnel_proc.poll() is not None:
        print("⚠️ সার্ভার বা টানেল বন্ধ হয়ে গেছে, এই সেলটা আবার চালান"); break